# 🏆 MCQ Competition — Complete Winning Pipeline

**Metric**: MAP@3 | **Task**: 5-choice MCQ | **Models**: TF-IDF · LR · SVM · LightGBM · XGBoost · CatBoost · Sentence-BERT · BGE · E5 · Cross-Encoder · DeBERTa

---

## Pipeline Overview
1. EDA + Visualization
2. Feature Engineering (TF-IDF, BM25, n-gram, length, readability)
3. Classical ML with 5-Fold Stratified CV
4. Sentence Transformer bi-encoders (MPNet, BGE, E5)
5. Cross-Encoder scoring (BGE-Reranker)
6. DeBERTa fine-tuning with MCQ head
7. Optuna ensemble weight optimization
8. MAP@3-optimized submission generation

In [ ]:
# MCQ Pipeline - Updated imports and configurations
# ============================================================
# CELL 1: SETUP & IMPORTS
# ============================================================
import os, gc, re, sys, time, json, warnings, random
import numpy as np
import pandas as pd
from pathlib import Path
from copy import deepcopy
from typing import List, Dict, Tuple, Optional, Any

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    os.environ['PYTHONHASHSEED'] = str(s)
    try:
        import torch
        torch.manual_seed(s); torch.cuda.manual_seed_all(s)
        torch.backends.cudnn.deterministic = True
    except: pass
set_seed()

try:
    import torch
    DEVICE  = torch.device('cuda' if torch.cuda.is_available() else
                           'mps'  if torch.backends.mps.is_available() else 'cpu')
    USE_AMP = DEVICE.type == 'cuda'
    HAS_TORCH = True
    print(f'✅ PyTorch {torch.__version__} | Device: {DEVICE} | AMP: {USE_AMP}')
except ImportError:
    DEVICE = 'cpu'; USE_AMP = False; HAS_TORCH = False
    print('⚠️  PyTorch not available')

BASE_DIR   = Path('/Users/deepanshusingh/Desktop/DL GEN AI')
TRAIN_PATH = BASE_DIR / 'train (1).csv'
TEST_PATH  = BASE_DIR / 'test (1).csv'
OUT_DIR    = BASE_DIR / 'outputs'
MODEL_DIR  = BASE_DIR / 'saved_models'
OUT_DIR.mkdir(exist_ok=True); MODEL_DIR.mkdir(exist_ok=True)

OPTION_COLS  = ['A','B','C','D','E']
LABEL_TO_IDX = {k:i for i,k in enumerate(OPTION_COLS)}
IDX_TO_LABEL = {i:k for k,i in LABEL_TO_IDX.items()}
print(f'📂 BASE: {BASE_DIR}')

In [ ]:
# ============================================================
# CELL 2: MAP@3 METRIC
# ============================================================
def apk(actual: int, predicted: List[int], k: int = 3) -> float:
    if len(predicted) > k: predicted = predicted[:k]
    score, hits = 0.0, 0
    for i, p in enumerate(predicted):
        if p == actual and p not in predicted[:i]:
            hits += 1; score += hits / (i + 1.0)
    return score

def mapk(actuals: List[int], preds: List[List[int]], k: int = 3) -> float:
    return float(np.mean([apk(a, p, k) for a, p in zip(actuals, preds)]))

def scores_to_top3(scores: np.ndarray) -> List[List[int]]:
    return [np.argsort(-row)[:3].tolist() for row in scores]

def labels_to_top3_str(scores: np.ndarray) -> List[str]:
    return [' '.join(IDX_TO_LABEL[i] for i in np.argsort(-row)[:3]) for row in scores]

def normalize_scores(scores: np.ndarray) -> np.ndarray:
    mn = scores.min(axis=1, keepdims=True)
    mx = scores.max(axis=1, keepdims=True)
    return (scores - mn) / (mx - mn + 1e-9)

# Quick sanity check
assert apk(0, [0, 1, 2]) == 1.0
assert apk(0, [1, 0, 2]) == 0.5
assert apk(0, [1, 2, 0]) == pytest_approx_manual = abs(apk(0,[1,2,0]) - 1/3) < 1e-6
print(f'✅ MAP@3 metric OK | Perfect score: {mapk([0],[[[0,1,2]]]):.3f}')

In [ ]:
# ============================================================
# CELL 3: DATA LOADING
# ============================================================
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)
train_df.columns = [c.strip() for c in train_df.columns]
test_df.columns  = [c.strip() for c in test_df.columns]
train_df['label'] = train_df['answer'].map(LABEL_TO_IDX)

print(f'Train: {train_df.shape} | Test: {test_df.shape}')
print(f'Missing (train): {train_df.isnull().sum().sum()}')
print(f'Missing (test) : {test_df.isnull().sum().sum()}')
print()
print('Answer distribution:')
dist  = train_df['answer'].value_counts().sort_index()
total = len(train_df)
for opt, cnt in dist.items():
    bar = '█' * int(cnt / total * 50)
    print(f'  {opt}: {bar:<52} {cnt:>4} ({cnt/total*100:.1f}%)')
train_df.head(3)

In [ ]:
# ============================================================
# CELL 4: EDA VISUALIZATIONS
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

plt.rcParams.update({
    'figure.facecolor': '#0f0f1a', 'axes.facecolor': '#1a1a2e',
    'axes.edgecolor': '#444',      'text.color': '#e0e0e0',
    'axes.labelcolor': '#e0e0e0',  'xtick.color': '#aaa',
    'ytick.color': '#aaa',         'grid.color': '#333', 'grid.alpha': 0.4,
})
PALETTE = ['#7c3aed','#06b6d4','#f59e0b','#10b981','#ef4444']

fig = plt.figure(figsize=(20, 16), facecolor='#0f0f1a')
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.5, wspace=0.35)

# 1. Answer distribution
ax1 = fig.add_subplot(gs[0, 0])
counts = train_df['answer'].value_counts().sort_index()
bars = ax1.bar(counts.index, counts.values, color=PALETTE, edgecolor='#222')
for bar, v in zip(bars, counts.values):
    ax1.text(bar.get_x()+bar.get_width()/2, v+3, str(v),
             ha='center', color='#e0e0e0', fontsize=9)
ax1.set_title('Answer Distribution', color='#7c3aed', fontweight='bold')
ax1.set_ylim(0, counts.max()*1.2)

# 2. Prompt length histogram
ax2 = fig.add_subplot(gs[0, 1])
pl = train_df['prompt'].str.len()
ax2.hist(pl, bins=40, color='#06b6d4', edgecolor='#0f0f1a', alpha=0.85)
ax2.axvline(pl.mean(), color='#f59e0b', linestyle='--', label=f'Mean={pl.mean():.0f}')
ax2.set_title('Prompt Length (chars)', color='#06b6d4', fontweight='bold')
ax2.legend(fontsize=8)

# 3. Option lengths by label
ax3 = fig.add_subplot(gs[0, 2])
for i, opt in enumerate(OPTION_COLS):
    ax3.hist(train_df[opt].str.len(), bins=30, alpha=0.55,
             color=PALETTE[i], label=opt)
ax3.set_title('Option Length (chars)', color='#f59e0b', fontweight='bold')
ax3.legend(fontsize=8)

# 4. Correct vs wrong option length
ax4 = fig.add_subplot(gs[1, 0])
correct_lens, wrong_lens = [], []
for _, row in train_df.iterrows():
    correct_lens.append(len(str(row[row['answer']])))
    wrong_lens.extend([len(str(row[c])) for c in OPTION_COLS if c != row['answer']])
ax4.hist(correct_lens, bins=30, alpha=0.75, color='#10b981', label='Correct')
ax4.hist(wrong_lens,   bins=30, alpha=0.45, color='#ef4444', label='Wrong')
ax4.set_title('Correct vs Wrong Option Lengths', color='#10b981', fontweight='bold')
ax4.legend(fontsize=8)

# 5. Word count boxplot by option
ax5 = fig.add_subplot(gs[1, 1])
wc_data = [train_df[opt].str.split().str.len() for opt in OPTION_COLS]
bp = ax5.boxplot(wc_data, labels=OPTION_COLS, patch_artist=True,
                  medianprops=dict(color='white', linewidth=2))
for patch, color in zip(bp['boxes'], PALETTE):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax5.set_title('Option Word Count Boxplot', color='#f59e0b', fontweight='bold')

# 6. Position bias
ax6 = fig.add_subplot(gs[1, 2])
bias_vals = [(train_df['answer']==opt).sum()/total*100 - 20
             for opt in OPTION_COLS]
colors = ['#10b981' if v >= 0 else '#ef4444' for v in bias_vals]
ax6.bar(OPTION_COLS, bias_vals, color=colors, edgecolor='#222')
ax6.axhline(0, color='white', linestyle='--', linewidth=1)
ax6.set_title('Answer Position Bias (% vs uniform)', color='#7c3aed', fontweight='bold')
ax6.set_ylabel('Bias (%)')

# 7. Duplicate prompts
ax7 = fig.add_subplot(gs[2, 0])
dup = train_df['prompt'].duplicated().sum()
ax7.pie([total-dup, dup], labels=['Unique','Duplicated'],
         colors=['#10b981','#ef4444'], autopct='%1.1f%%',
         textprops={'color':'#e0e0e0'})
ax7.set_title(f'Duplicate Prompts ({dup}/{total})', color='#06b6d4', fontweight='bold')

# 8. Correct option length vs avg other options
ax8 = fig.add_subplot(gs[2, 1:])
ratios = []
for _, row in train_df.iterrows():
    correct_len = len(str(row[row['answer']]))
    other_lens  = [len(str(row[c])) for c in OPTION_COLS if c != row['answer']]
    ratios.append(correct_len / (np.mean(other_lens) + 1e-9))
ax8.hist(ratios, bins=40, color='#7c3aed', edgecolor='#0f0f1a', alpha=0.85)
ax8.axvline(1.0, color='white', linestyle='--', linewidth=1.5, label='Equal length')
ax8.axvline(np.mean(ratios), color='#f59e0b', linestyle='--',
             label=f'Mean ratio={np.mean(ratios):.2f}')
ax8.set_title('Correct Option Length / Avg Wrong Option Length', color='#7c3aed', fontweight='bold')
ax8.legend(fontsize=9)

fig.suptitle('📊 MCQ Competition — EDA Dashboard', fontsize=16,
             color='white', fontweight='bold', y=1.01)
plt.savefig(OUT_DIR/'eda_dashboard.png', bbox_inches='tight',
            facecolor='#0f0f1a', dpi=150)
plt.show()

print(f'\n💡 Key Insights:')
print(f'  • Correct answers are {np.mean(correct_lens)/np.mean(wrong_lens):.2f}x longer on avg than wrong answers')
print(f'  • Answer B has highest frequency ({(train_df["answer"]=="B").sum()/total*100:.1f}%) — position bias!')
print(f'  • {dup}/{total} prompts duplicated — deduplication may help or cause leakage')
print(f'  • {(train_df["prompt"].str.len() > 200).sum()} prompts longer than 200 chars')

In [ ]:
# ============================================================
# CELL 5: FEATURE ENGINEERING
# ============================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
import textstat
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
STOPWORDS = set(stopwords.words('english'))

def tokenize(text: str) -> List[str]:
    return re.sub(r'[^a-z0-9\s]', ' ', str(text).lower()).split()

def ngram_overlap(t1: str, t2: str, n: int = 2) -> float:
    def ngrams(toks, n): return set(zip(*[toks[i:] for i in range(n)]))
    a, b = tokenize(t1), tokenize(t2)
    ga, gb = ngrams(a, n), ngrams(b, n)
    if not ga and not gb: return 0.0
    return len(ga & gb) / (len(ga | gb) + 1e-9)

def keyword_overlap(t1: str, t2: str) -> float:
    a = set(tokenize(t1)) - STOPWORDS
    b = set(tokenize(t2)) - STOPWORDS
    if not a and not b: return 0.0
    return len(a & b) / (len(a | b) + 1e-9)

NEG_WORDS = {'not','no','never','neither','nor','without','cannot','isn\'t','don\'t','won\'t'}

def build_features(df: pd.DataFrame, tfidf_vec=None, fit: bool=False):
    all_texts = [str(r['prompt']) + ' ' + str(r[c]) for _,r in df.iterrows() for c in OPTION_COLS]
    if fit or tfidf_vec is None:
        tfidf_vec = TfidfVectorizer(ngram_range=(1,2), max_features=50000,
                                     sublinear_tf=True, min_df=2, max_df=0.95)
        tfidf_vec.fit(all_texts)
    rows = []
    for _, row in df.iterrows():
        prompt   = str(row['prompt'])
        opts     = [str(row[c]) for c in OPTION_COLS]
        bm25     = BM25Okapi([tokenize(o) for o in opts])
        bm25s    = bm25.get_scores(tokenize(prompt))
        pvec     = tfidf_vec.transform([prompt])
        opt_lens = [len(o) for o in opts]
        mean_len = np.mean(opt_lens)
        sorted_l = sorted(opt_lens, reverse=True)
        for i, opt in enumerate(opts):
            ovec = tfidf_vec.transform([opt])
            tc   = float(cosine_similarity(pvec, ovec)[0,0])
            br   = 5 - int(np.sum(bm25s < bm25s[i]))
            rows.append([
                len(prompt), len(opt), len(opt.split()),
                len(opt)/(len(prompt)+1), len(opt.split())/(len(prompt.split())+1),
                int(len(opt)>200), int(len(opt)<30),
                keyword_overlap(prompt, opt),
                ngram_overlap(prompt, opt, 2),
                ngram_overlap(prompt, opt, 3),
                i, sorted_l.index(len(opt)),
                int(bool(re.match(r'^\d', opt))),
                int(opt[0].isupper()) if opt else 0,
                int(' '.join(opt.split()[:4]).lower() in prompt.lower()),
                len(opt)/(mean_len+1e-9),
                len(re.findall(r'\b\d+\.?\d*\b', opt)),
                int(bool(NEG_WORDS & set(tokenize(opt)))),
                tc, float(bm25s[i]), br,
                textstat.flesch_reading_ease(opt),
                textstat.flesch_kincaid_grade(opt),
            ])
    return np.array(rows, dtype=np.float32), tfidf_vec

print('Building features (train) ...')
X_train, tfidf_vec = build_features(train_df, fit=True)
print('Building features (test) ...')
X_test, _          = build_features(test_df, tfidf_vec=tfidf_vec)

from joblib import dump, load
dump(tfidf_vec, MODEL_DIR/'tfidf_vectorizer.joblib')
np.save(OUT_DIR/'X_train.npy', X_train)
np.save(OUT_DIR/'X_test.npy',  X_test)

y_flat    = np.array([1.0 if OPTION_COLS[i%5]==train_df.iloc[i//5]['answer'] else 0.0 for i in range(len(train_df)*5)])
label_arr = train_df['label'].values

print(f'\nX_train: {X_train.shape} | X_test: {X_test.shape}')
print(f'y_flat: {y_flat.shape} | pos ratio: {y_flat.mean():.4f}')
print(f'Features: prompt_len, option_len, option_words, char_ratio, word_ratio, is_long, is_short, keyword_overlap, bigram_overlap, trigram_overlap, position, len_rank, starts_num, starts_cap, option_in_prompt, len_vs_mean, num_numbers, has_negation, tfidf_cosine, bm25_score, bm25_rank, flesch_ease, flesch_grade')

In [ ]:
# ============================================================
# CELL 6: CLASSICAL ML — 5-FOLD CV
# ============================================================
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import lightgbm as lgb
import xgboost as xgb
import catboost as cb

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def option_probs(model, X_flat, n=5):
    if hasattr(model, 'predict_proba'): p = model.predict_proba(X_flat)[:,1]
    else:
        p = model.decision_function(X_flat); p = (p-p.min())/(p.ptp()+1e-9)
    return p.reshape(-1, n)

MODELS = {
    'lr'  : Pipeline([('sc', StandardScaler()),
                       ('clf', LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', random_state=SEED))]),
    'svm' : Pipeline([('sc', StandardScaler()),
                       ('clf', CalibratedClassifierCV(LinearSVC(C=0.5, max_iter=2000, random_state=SEED), cv=3))]),
    'lgbm': lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, num_leaves=63,
                                 subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1,
                                 min_child_samples=10, random_state=SEED, verbose=-1),
    'xgb' : xgb.XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=5, subsample=0.8,
                                colsample_bytree=0.8, eval_metric='logloss', random_state=SEED, verbosity=0),
    'cat' : cb.CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6, l2_leaf_reg=3.0,
                                   subsample=0.8, random_seed=SEED, verbose=0),
}

all_oof  = {}
all_test = {}

for mname, model in MODELS.items():
    print(f'\n{"─"*50}\n  Training: {mname.upper()}')
    oof_prob   = np.zeros((len(train_df), 5))
    test_folds = np.zeros((len(test_df), 5, 5))
    fold_sc    = []

    for fold, (tq, vq) in enumerate(skf.split(np.zeros(len(train_df)), label_arr)):
        ti = np.concatenate([np.arange(q*5,q*5+5) for q in tq])
        vi = np.concatenate([np.arange(q*5,q*5+5) for q in vq])
        Xtr, ytr = X_train[ti], y_flat[ti]
        Xvl, yvl = X_train[vi], y_flat[vi]

        if mname == 'lgbm':
            model.fit(Xtr, ytr, eval_set=[(Xvl,yvl)],
                      callbacks=[lgb.early_stopping(50,verbose=False),lgb.log_evaluation(-1)])
        elif mname == 'xgb':
            model.fit(Xtr, ytr, eval_set=[(Xvl,yvl)], verbose=False, early_stopping_rounds=50)
        elif mname == 'cat':
            model.fit(Xtr, ytr, eval_set=(Xvl,yvl), early_stopping_rounds=50, verbose=False)
        else:
            model.fit(Xtr, ytr)

        oof_prob[vq]          = option_probs(model, Xvl)
        test_folds[:,:,fold]  = option_probs(model, X_test)
        fm3 = mapk(label_arr[vq].tolist(), scores_to_top3(oof_prob[vq]))
        fold_sc.append(fm3)
        print(f'  Fold {fold+1}  MAP@3 = {fm3:.4f}')

    cv = np.mean(fold_sc)
    print(f'  ► {mname.upper()} CV MAP@3 = {cv:.4f} ± {np.std(fold_sc):.4f}')
    all_oof[mname]  = normalize_scores(oof_prob)
    all_test[mname] = normalize_scores(test_folds.mean(axis=2))
    dump(model, MODEL_DIR/f'{mname}_full_model.joblib')

print('\n📊 Summary:')
for n,p in all_oof.items():
    sc = mapk(label_arr.tolist(), scores_to_top3(p))
    print(f'  {n:<10} MAP@3 = {sc:.4f}')

In [ ]:
# ============================================================
# CELL 7: FEATURE IMPORTANCE PLOTS
# ============================================================
FEAT_NAMES = [
    'prompt_len','option_len','option_words','char_ratio','word_ratio',
    'is_long','is_short','keyword_overlap','bigram_overlap','trigram_overlap',
    'position','len_rank','starts_num','starts_cap','option_in_prompt',
    'len_vs_mean','num_numbers','has_negation','tfidf_cosine',
    'bm25_score','bm25_rank','flesch_ease','flesch_grade'
]

for mname in ['lgbm','xgb','cat']:
    m = MODELS[mname]
    if not hasattr(m, 'feature_importances_'): continue
    imp = m.feature_importances_
    df_imp = pd.DataFrame({'feature':FEAT_NAMES,'importance':imp}).nlargest(15,'importance')
    fig, ax = plt.subplots(figsize=(10, 5), facecolor='#0f0f1a')
    ax.set_facecolor('#1a1a2e')
    ax.barh(df_imp['feature'], df_imp['importance'], color='#7c3aed', alpha=0.85)
    ax.set_title(f'Feature Importance — {mname.upper()}', color='#7c3aed', fontweight='bold')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(OUT_DIR/f'feat_imp_{mname}.png', facecolor='#0f0f1a', dpi=120)
    plt.show()

In [ ]:
# ============================================================
# CELL 8: SENTENCE TRANSFORMERS (bi-encoders)
# ============================================================
from tqdm.auto import tqdm
try:
    from sentence_transformers import SentenceTransformer
    HAS_ST = True
except ImportError:
    HAS_ST = False; print('⚠️  sentence-transformers not available')

def compute_st_scores(df, model_id, batch_size=64, cache_prefix=''):
    cache = OUT_DIR / f'{cache_prefix}.npy'
    if cache.exists():
        print(f'  ✓ Loaded from cache: {cache.name}')
        return np.load(cache)
    if not HAS_ST:
        return np.zeros((len(df), 5))
    print(f'  Embedding: {model_id}')
    m = SentenceTransformer(model_id, device=str(DEVICE))
    p_embs = m.encode(df['prompt'].tolist(), batch_size=batch_size,
                       show_progress_bar=True, normalize_embeddings=True)
    flat   = [str(row[c]) for _,row in df.iterrows() for c in OPTION_COLS]
    o_embs = m.encode(flat, batch_size=batch_size,
                       show_progress_bar=True, normalize_embeddings=True)
    o_3d   = o_embs.reshape(len(df), 5, -1)
    scores = np.einsum('nd,nkd->nk', p_embs, o_3d)
    np.save(cache, scores)
    del m; gc.collect()
    return scores

ST_CONFIGS = [
    ('mpnet',  'sentence-transformers/all-mpnet-base-v2'),
    ('bge',    'BAAI/bge-large-en-v1.5'),
    ('e5',     'intfloat/e5-large-v2'),
    ('minilm', 'sentence-transformers/all-MiniLM-L12-v2'),
]

for key, mid in ST_CONFIGS:
    try:
        s_tr = compute_st_scores(train_df, mid, cache_prefix=f'st_train_{key}')
        s_te = compute_st_scores(test_df,  mid, cache_prefix=f'st_test_{key}')
        all_oof[f'st_{key}']  = normalize_scores(s_tr)
        all_test[f'st_{key}'] = normalize_scores(s_te)
        sc = mapk(label_arr.tolist(), scores_to_top3(all_oof[f'st_{key}']))
        print(f'  st_{key} MAP@3 = {sc:.4f}')
    except Exception as e:
        print(f'  ⚠️ {key}: {e}')

In [ ]:
# ============================================================
# CELL 9: CROSS-ENCODERS
# ============================================================
try:
    from sentence_transformers import CrossEncoder
    HAS_CE = True
except ImportError:
    HAS_CE = False; print('⚠️  CrossEncoder not available')

def compute_ce_scores(df, model_id, batch_size=32, cache_prefix=''):
    cache = OUT_DIR / f'{cache_prefix}.npy'
    if cache.exists():
        print(f'  ✓ Loaded from cache: {cache.name}')
        return np.load(cache)
    if not HAS_CE:
        return np.zeros((len(df), 5))
    print(f'  Cross-encoding: {model_id}')
    ce = CrossEncoder(model_id, device=str(DEVICE), max_length=512)
    all_scores = []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        pairs  = [(str(row['prompt']), str(row[c])) for c in OPTION_COLS]
        scores = ce.predict(pairs, batch_size=batch_size, show_progress_bar=False)
        all_scores.append(scores)
    result = np.array(all_scores)
    np.save(cache, result)
    del ce; gc.collect()
    return result

CE_CONFIGS = [
    ('ce_minilm', 'cross-encoder/ms-marco-MiniLM-L-12-v2'),
    ('bge_rerank','BAAI/bge-reranker-base'),
]

for key, mid in CE_CONFIGS:
    try:
        s_tr = compute_ce_scores(train_df, mid, cache_prefix=f'ce_train_{key}')
        s_te = compute_ce_scores(test_df,  mid, cache_prefix=f'ce_test_{key}')
        all_oof[key]  = normalize_scores(s_tr)
        all_test[key] = normalize_scores(s_te)
        sc = mapk(label_arr.tolist(), scores_to_top3(all_oof[key]))
        print(f'  {key} MAP@3 = {sc:.4f}')
    except Exception as e:
        print(f'  ⚠️ {key}: {e}')

In [ ]:
# ============================================================
# CELL 10: DeBERTa FINE-TUNING (MCQ HEAD)
# ============================================================
DEBERTA_MODEL_ID    = 'microsoft/deberta-v3-base'  # → deberta-v3-large for best score
DEBERTA_MAX_LEN     = 256
DEBERTA_BATCH_SIZE  = 4
DEBERTA_ACCUM_STEPS = 8    # effective batch = 32
DEBERTA_EPOCHS      = 3
DEBERTA_LR          = 2e-5
DEBERTA_WD          = 0.01
DEBERTA_WARMUP      = 0.1
LABEL_SMOOTHING     = 0.1

if HAS_TORCH:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
    from torch.optim import AdamW
    from torch.cuda.amp import autocast, GradScaler
    from transformers import (AutoTokenizer, AutoModelForMultipleChoice,
                               get_cosine_schedule_with_warmup)

    class MCQDataset(Dataset):
        def __init__(self, df, tokenizer, max_len=256, has_labels=True):
            self.df = df.reset_index(drop=True)
            self.tok = tokenizer
            self.max_len = max_len
            self.has_labels = has_labels
        def __len__(self): return len(self.df)
        def __getitem__(self, idx):
            row = self.df.iloc[idx]
            encs = [self.tok(str(row['prompt']), str(row[c]),
                             max_length=self.max_len, truncation=True,
                             padding='max_length', return_tensors='pt')
                    for c in OPTION_COLS]
            item = {
                'input_ids':      torch.stack([e['input_ids'].squeeze(0)      for e in encs]),
                'attention_mask': torch.stack([e['attention_mask'].squeeze(0) for e in encs]),
            }
            if 'token_type_ids' in encs[0]:
                item['token_type_ids'] = torch.stack([e['token_type_ids'].squeeze(0) for e in encs])
            if self.has_labels:
                item['labels'] = torch.tensor(LABEL_TO_IDX[str(row['answer'])], dtype=torch.long)
            return item

    def label_smoothed_ce(logits, labels, eps=0.1):
        n_cls = logits.size(-1)
        lp    = F.log_softmax(logits, dim=-1)
        nll   = F.nll_loss(lp, labels, reduction='mean')
        smooth = -lp.mean(dim=-1).mean()
        return (1-eps)*nll + eps*smooth

    tokenizer = AutoTokenizer.from_pretrained(DEBERTA_MODEL_ID)
    deberta_oof   = np.zeros((len(train_df), 5))
    deberta_test  = np.zeros((len(test_df), 5, 5))

    for fold, (tr_idx, vl_idx) in enumerate(skf.split(train_df, label_arr)):
        print(f'\n── DeBERTa Fold {fold+1}/5 ─────────────────────────────')
        tr_ds = MCQDataset(train_df.iloc[tr_idx], tokenizer, DEBERTA_MAX_LEN)
        vl_ds = MCQDataset(train_df.iloc[vl_idx], tokenizer, DEBERTA_MAX_LEN)
        te_ds = MCQDataset(test_df, tokenizer, DEBERTA_MAX_LEN, has_labels=False)
        tr_ldr = DataLoader(tr_ds, batch_size=DEBERTA_BATCH_SIZE, shuffle=True)
        vl_ldr = DataLoader(vl_ds, batch_size=DEBERTA_BATCH_SIZE*2, shuffle=False)
        te_ldr = DataLoader(te_ds, batch_size=DEBERTA_BATCH_SIZE*2, shuffle=False)

        model = AutoModelForMultipleChoice.from_pretrained(DEBERTA_MODEL_ID).to(DEVICE)
        opt   = AdamW(model.parameters(), lr=DEBERTA_LR, weight_decay=DEBERTA_WD)
        total = len(tr_ldr)//DEBERTA_ACCUM_STEPS * DEBERTA_EPOCHS
        sched = get_cosine_schedule_with_warmup(opt, int(total*DEBERTA_WARMUP), total)
        scaler = GradScaler() if USE_AMP else None

        best_map3, best_state = 0.0, None

        for epoch in range(DEBERTA_EPOCHS):
            model.train(); opt.zero_grad(); ep_loss = 0.0
            for step, batch in enumerate(tqdm(tr_ldr, desc=f'Ep{epoch+1} Train', leave=False)):
                labs = batch.pop('labels').to(DEVICE)
                batch = {k:v.to(DEVICE) for k,v in batch.items()}
                if USE_AMP:
                    with autocast():
                        out  = model(**batch)
                        loss = label_smoothed_ce(out.logits, labs, LABEL_SMOOTHING)
                        loss = loss / DEBERTA_ACCUM_STEPS
                    scaler.scale(loss).backward()
                    if (step+1) % DEBERTA_ACCUM_STEPS == 0:
                        scaler.unscale_(opt)
                        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad()
                else:
                    out  = model(**batch)
                    loss = label_smoothed_ce(out.logits, labs, LABEL_SMOOTHING) / DEBERTA_ACCUM_STEPS
                    loss.backward()
                    if (step+1) % DEBERTA_ACCUM_STEPS == 0:
                        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        opt.step(); sched.step(); opt.zero_grad()
                ep_loss += loss.item() * DEBERTA_ACCUM_STEPS

            # Validation
            model.eval()
            vl_logits, vl_labs = [], []
            with torch.no_grad():
                for batch in tqdm(vl_ldr, desc='Val', leave=False):
                    labs = batch.pop('labels').to(DEVICE)
                    batch = {k:v.to(DEVICE) for k,v in batch.items()}
                    out = model(**batch) if not USE_AMP else model(**batch)
                    vl_logits.append(out.logits.float().cpu())
                    vl_labs.append(labs.cpu())
            vl_probs  = F.softmax(torch.cat(vl_logits), dim=-1).numpy()
            vl_labels = torch.cat(vl_labs).numpy()
            vmap3 = mapk(vl_labels.tolist(), scores_to_top3(vl_probs))
            print(f'  Ep{epoch+1}: loss={ep_loss/len(tr_ldr):.4f}  val_MAP@3={vmap3:.4f}')
            if vmap3 > best_map3:
                best_map3, best_state = vmap3, deepcopy(model.state_dict())

        if best_state: model.load_state_dict(best_state)
        model.eval()

        # OOF
        oof_logits = []
        with torch.no_grad():
            for batch in tqdm(vl_ldr, desc='OOF', leave=False):
                batch.pop('labels', None)
                batch = {k:v.to(DEVICE) for k,v in batch.items()}
                oof_logits.append(model(**batch).logits.float().cpu())
        deberta_oof[vl_idx] = F.softmax(torch.cat(oof_logits), dim=-1).numpy()

        # Test
        te_logits = []
        with torch.no_grad():
            for batch in tqdm(te_ldr, desc='Test', leave=False):
                batch.pop('labels', None)
                batch = {k:v.to(DEVICE) for k,v in batch.items()}
                te_logits.append(model(**batch).logits.float().cpu())
        deberta_test[:,:,fold] = F.softmax(torch.cat(te_logits), dim=-1).numpy()

        model.save_pretrained(MODEL_DIR/f'deberta_fold{fold+1}')
        tokenizer.save_pretrained(MODEL_DIR/f'deberta_fold{fold+1}')
        del model; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        print(f'  Fold {fold+1} MAP@3 = {mapk(label_arr[vl_idx].tolist(), scores_to_top3(deberta_oof[vl_idx])):.4f}')

    deberta_test_avg = deberta_test.mean(axis=2)
    db_map3 = mapk(label_arr.tolist(), scores_to_top3(deberta_oof))
    print(f'\n🏆 DeBERTa CV MAP@3 = {db_map3:.4f}')
    all_oof['deberta']  = normalize_scores(deberta_oof)
    all_test['deberta'] = normalize_scores(deberta_test_avg)
    np.save(OUT_DIR/'deberta_oof.npy',  deberta_oof)
    np.save(OUT_DIR/'deberta_test.npy', deberta_test_avg)
else:
    print('⚠️  DeBERTa skipped (PyTorch unavailable)')

In [ ]:
# ============================================================
# CELL 11: ALL MODELS SUMMARY
# ============================================================
print('='*55)
print('  INDIVIDUAL MODEL SCORES (CV MAP@3)')
print('='*55)
model_scores = {}
for name, preds in all_oof.items():
    sc = mapk(label_arr.tolist(), scores_to_top3(preds))
    model_scores[name] = sc
    bar = '█' * int(sc*50)
    print(f'  {name:<25} {bar:<55} {sc:.4f}')
print('='*55)

In [ ]:
# ============================================================
# CELL 12: OPTUNA ENSEMBLE WEIGHT OPTIMIZATION
# ============================================================
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

MODEL_NAMES = list(all_oof.keys())

def objective(trial):
    w = np.array([trial.suggest_float(f'w_{n}', 0.0, 1.0) for n in MODEL_NAMES])
    w = w / (w.sum() + 1e-9)
    ens = sum(w[i] * all_oof[MODEL_NAMES[i]] for i in range(len(MODEL_NAMES)))
    return mapk(label_arr.tolist(), scores_to_top3(ens))

study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner()
)
print('Running Optuna (200 trials) ...')
study.optimize(objective, n_trials=200, show_progress_bar=True)

best_params = study.best_trial.params
best_map3   = study.best_trial.value
print(f'\n🏆 Best ensemble MAP@3 = {best_map3:.4f}')

# Extract and normalize weights
raw_w  = np.array([best_params[f'w_{n}'] for n in MODEL_NAMES])
norm_w = raw_w / (raw_w.sum() + 1e-9)

print(f'\n{"Model":<30} {"Weight":>8}')
print('-'*40)
for name, w in sorted(zip(MODEL_NAMES, norm_w), key=lambda x: -x[1]):
    print(f'  {name:<28} {w:>8.4f}')

In [ ]:
# ============================================================
# CELL 13: FINAL ENSEMBLE PREDICTIONS + ERROR ANALYSIS
# ============================================================
# Apply best weights
final_train = sum(norm_w[i] * all_oof[MODEL_NAMES[i]]  for i in range(len(MODEL_NAMES)))
final_test  = sum(norm_w[i] * all_test[MODEL_NAMES[i]] for i in range(len(MODEL_NAMES)))

oof_map3 = mapk(label_arr.tolist(), scores_to_top3(final_train))
print(f'\n✅ Final Ensemble OOF MAP@3 = {oof_map3:.4f}')

# ── Error Analysis ─────────────────────────────────────────────────────────────
top1_pred = np.argmax(final_train, axis=1)
top1_acc  = (top1_pred == label_arr).mean()
top3_acc  = np.mean([label_arr[i] in scores_to_top3(final_train)[i] for i in range(len(label_arr))])

print(f'\n{'='*50}')
print('  ERROR ANALYSIS')
print(f'{'='*50}')
print(f'  Top-1 Accuracy : {top1_acc:.4f} ({top1_acc*100:.2f}%)')
print(f'  Top-3 Accuracy : {top3_acc:.4f} ({top3_acc*100:.2f}%)')
print(f'  OOF MAP@3      : {oof_map3:.4f}')

# Confusion matrix
conf = np.zeros((5, 5), int)
for i, (t, p) in enumerate(zip(label_arr, top1_pred)):
    conf[t, p] += 1

fig, ax = plt.subplots(figsize=(7, 5), facecolor='#0f0f1a')
ax.set_facecolor('#1a1a2e')
im = ax.imshow(conf, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(5)); ax.set_yticks(range(5))
ax.set_xticklabels(OPTION_COLS, color='#e0e0e0')
ax.set_yticklabels(OPTION_COLS, color='#e0e0e0')
ax.set_xlabel('Predicted', color='#e0e0e0'); ax.set_ylabel('True', color='#e0e0e0')
ax.set_title('OOF Confusion Matrix', color='#7c3aed', fontweight='bold')
for i in range(5):
    for j in range(5):
        ax.text(j, i, str(conf[i,j]), ha='center', va='center',
                color='white' if conf[i,j] > conf.max()*0.5 else '#333', fontweight='bold')
plt.colorbar(im, ax=ax); plt.tight_layout()
plt.savefig(OUT_DIR/'confusion_matrix.png', facecolor='#0f0f1a', dpi=120)
plt.show()

# Confidence distribution
max_p = final_train.max(axis=1)
fig, ax = plt.subplots(figsize=(10, 4), facecolor='#0f0f1a')
ax.set_facecolor('#1a1a2e')
ax.hist(max_p, bins=40, color='#7c3aed', edgecolor='#0f0f1a', alpha=0.85)
ax.axvline(max_p.mean(), color='#f59e0b', linestyle='--', label=f'Mean={max_p.mean():.3f}')
ax.set_title('Prediction Confidence Distribution', color='#7c3aed', fontweight='bold')
ax.legend(); plt.tight_layout()
plt.savefig(OUT_DIR/'confidence_dist.png', facecolor='#0f0f1a', dpi=120)
plt.show()

In [ ]:
# ============================================================
# CELL 14: THRESHOLD OPTIMIZATION
# ============================================================
# Alpha sweep: how much weight to give DeBERTa vs rest
if 'deberta' in all_oof:
    alphas = np.linspace(0, 1, 21)
    rest   = np.mean([all_oof[k] for k in MODEL_NAMES if k != 'deberta'], axis=0)
    scores = [mapk(label_arr.tolist(),
                   scores_to_top3(a*all_oof['deberta'] + (1-a)*rest))
              for a in alphas]
    best_a = alphas[np.argmax(scores)]
    print(f'Best DeBERTa alpha = {best_a:.2f}  MAP@3 = {max(scores):.4f}')

    fig, ax = plt.subplots(figsize=(10, 4), facecolor='#0f0f1a')
    ax.set_facecolor('#1a1a2e')
    ax.plot(alphas, scores, 'o-', color='#7c3aed', linewidth=2)
    ax.axvline(best_a, color='#f59e0b', linestyle='--', label=f'Best α={best_a:.2f}')
    ax.set_xlabel('DeBERTa Weight', color='#e0e0e0')
    ax.set_ylabel('MAP@3', color='#e0e0e0')
    ax.set_title('DeBERTa α Sweep', color='#7c3aed', fontweight='bold')
    ax.legend(); plt.tight_layout()
    plt.savefig(OUT_DIR/'alpha_sweep.png', facecolor='#0f0f1a', dpi=120)
    plt.show()
else:
    print('DeBERTa not available for alpha sweep')

In [ ]:
# ============================================================
# CELL 15: GENERATE SUBMISSION
# ============================================================
predictions = labels_to_top3_str(final_test)
sub = pd.DataFrame({'ID': test_df['id'].values, 'Prediction': predictions})
sub.to_csv(OUT_DIR/'submission.csv', index=False)
sub.to_csv(BASE_DIR/'submission.csv', index=False)

print(f'✅ submission.csv saved!')
print(f'   Shape: {sub.shape}')
print()
print(sub.head(10).to_string(index=False))

# Verify format
assert sub.shape == (len(test_df), 2), 'Wrong submission shape'
assert all(sub['Prediction'].str.split().str.len() == 3), 'Not exactly 3 predictions'
valid_labels = set('A B C D E'.split())
for pred in sub['Prediction']:
    assert set(pred.split()).issubset(valid_labels), f'Invalid labels in: {pred}'
print('\n✅ Submission format verified!')

In [ ]:
# ============================================================
# CELL 16: FINAL REPORT
# ============================================================
print('='*65)
print('  FINAL PIPELINE REPORT')
print('='*65)
print(f'  Train samples       : {len(train_df)}')
print(f'  Test samples        : {len(test_df)}')
print(f'  Feature dimensions  : {X_train.shape[1]}')
print(f'  CV folds            : 5')
print(f'  Random seed         : {SEED}')
print(f'  Optuna trials       : 200')
print()
print(f'  {"Model":<30} {"MAP@3":>8}')
print(f'  {"-"*40}')
for n, s in sorted(model_scores.items(), key=lambda x: -x[1]):
    print(f'  {n:<30} {s:>8.4f}')
print(f'  {"-"*40}')
print(f'  {"FINAL ENSEMBLE":<30} {oof_map3:>8.4f}')
print('='*65)
print()
print('🎉 Pipeline complete!')
print(f'📄 submission.csv → {BASE_DIR}/submission.csv')
print(f'📊 EDA plots      → {OUT_DIR}/')
print(f'🔍 Models         → {MODEL_DIR}/')